この教材では、プログラムどうしが **ネットワーク越しに通信する** 仕組みを、手を動かして学びます。

- **HTTPとサーバ**: HTTPの仕組み → 自分でWebサーバを立てる → クライアントで叩く → 通信の正体（生のTCP）を覗く
- **リアルタイム通信**: HTTPの限界 → **WebSocket**（双方向・持続接続） → **WebRTC**（ブラウザ同士のP2P通信）

前回の「バイナリとセキュリティ」で学んだこと（バイト列・XOR・ハッシュ）が、ここで実際の通信の中に現れます。最後の総仕上げでは、**サーバと対戦する数当てbot** を作ります。これは次の「対戦ゲーム回」への布石です。

> このノートブックのサーバは、あなたのColab（ランタイム）の中で動き、同じランタイムから `localhost`（自分自身）に接続して試します。教室のみんなで相互接続して遊ぶのは、次の対戦回で扱います。

# HTTP とサーバ

## HTTP とは

Webの通信の大部分は **HTTP** というルール（プロトコル）で行われます。基本は **リクエスト（要求）** と **レスポンス（応答）** の一往復です。

- **メソッド**: 何をしたいか。`GET`（取得）、`POST`（送信）など
- **URL / パス**: どこに対して。`/hello` など
- **ステータスコード**: 結果。`200`（成功）、`404`（見つからない）、`500`（サーバ側エラー）など
- **ヘッダ**: 付随情報（データの種類 `Content-Type` など）と **ボディ**（本文データ）

まずは自分でサーバを立てて、この往復を体験します。

## 自分の Web サーバを立てる

Pythonの標準ライブラリ `http.server` で、小さなWebサーバを作れます。`do_GET` / `do_POST` に「リクエストが来たら何を返すか」を書きます。

サーバは動き続ける（待ち受ける）ので、ノートブックが固まらないよう **別スレッド** で起動します。下のセルを実行すると、`localhost` の空きポートでサーバが立ち上がります。

In [ ]:
import json, threading, time
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

class MyHandler(BaseHTTPRequestHandler):
    def _send_json(self, code, obj):
        body = json.dumps(obj, ensure_ascii=False).encode()
        self.send_response(code)                       # ステータスコード
        self.send_header("Content-Type", "application/json; charset=utf-8")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)                         # ボディを書き込む

    def do_GET(self):
        if self.path == "/hello":
            self._send_json(200, {"message": "こんにちは"})
        else:
            self._send_json(404, {"error": "not found"})

    def do_POST(self):
        length = int(self.headers.get("Content-Length", 0))
        data = json.loads(self.rfile.read(length) or b"{}")   # 送られてきたJSON
        self._send_json(200, {"あなたが送ったデータ": data})

    def log_message(self, *args):
        pass   # アクセスログを出さない（授業用に静かに）

server = ThreadingHTTPServer(("127.0.0.1", 0), MyHandler)  # 0 = 空きポートを自動選択
PORT = server.server_address[1]
BASE = "http://127.0.0.1:{}".format(PORT)
threading.Thread(target=server.serve_forever, daemon=True).start()
time.sleep(0.3)
print("サーバ起動:", BASE)

## 練習問題1: サーバに新しいルートを追加する

サーバに `GET /double?n=5` のようなアクセスが来たら、`n` を **2倍** にして返すルートを作ってください。下の `do_GET` の `____` を埋めます。

**ヒント**: クエリ文字列 `?n=5` は `urllib.parse` で取り出せます（コードに用意済み）。`n` を2倍して `{"result": ...}` を返します。

**期待される動作**: `GET /double?n=5` → `{"result": 10}`

In [ ]:
import json, threading, time
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import urlparse, parse_qs
import urllib.request

class DoubleHandler(BaseHTTPRequestHandler):
    def do_GET(self):
        parsed = urlparse(self.path)          # 例: /double?n=5
        params = parse_qs(parsed.query)       # 例: {"n": ["5"]}
        n = int(params.get("n", ["0"])[0])

        result = ____                          # n を2倍にする

        body = json.dumps({"result": result}).encode()
        self.send_response(200)
        self.send_header("Content-Type", "application/json")
        self.end_headers()
        self.wfile.write(body)
    def log_message(self, *a): pass

srv2 = ThreadingHTTPServer(("127.0.0.1", 0), DoubleHandler)
p2 = srv2.server_address[1]
threading.Thread(target=srv2.serve_forever, daemon=True).start()
time.sleep(0.3)

# 動作確認
print(urllib.request.urlopen("http://127.0.0.1:{}/double?n=5".format(p2)).read().decode())  # {"result": 10}

## クライアントでサーバを叩く

サーバに **リクエストを送る側** がクライアントです。ここでは標準ライブラリ `urllib.request` を使います（実務では `requests` ライブラリもよく使われます）。

- `GET /hello` … 挨拶を取得
- `POST` … JSONを送って、返事を受け取る

ステータスコードやヘッダも確認してみましょう。

In [ ]:
import urllib.request, json

# --- GET ---
with urllib.request.urlopen(BASE + "/hello") as res:
    print("ステータス:", res.status)                 # 200
    print("Content-Type:", res.headers["Content-Type"])
    print("ボディ:", res.read().decode())            # {"message": "こんにちは"}

# --- POST（JSONを送る）---
payload = json.dumps({"name": "太郎", "score": 90}).encode()
req = urllib.request.Request(BASE + "/", data=payload,
                            headers={"Content-Type": "application/json"}, method="POST")
with urllib.request.urlopen(req) as res:
    print("POSTの返事:", res.read().decode())

## 練習問題2: クライアントでJSONを送り、返事を読む

教材のサーバ（`BASE`）に、`POST` で `{"name": "花子"}` を送り、返ってきたJSONを **辞書として** 受け取って表示してください。

**ヒント**: 送るときは `json.dumps(...).encode()`。受け取ったら `json.loads(res.read())` で辞書に戻せます。

In [ ]:
import urllib.request, json

payload = ____   # {"name": "花子"} をJSON文字列にして encode する
req = urllib.request.Request(BASE + "/", data=payload,
                            headers={"Content-Type": "application/json"}, method="POST")
with urllib.request.urlopen(req) as res:
    result = ____   # res.read() をJSON（辞書）として読み込む

print(result)

## HTTP の正体は「ただのテキスト」

HTTPは、実は **TCP という土管の上を流れる、決まった書式のテキスト** にすぎません。`socket` で生のTCP接続を開き、HTTPリクエストを **手で組み立てて** 送ってみると、それがよく分かります。

リクエストの1行目は `メソッド パス プロトコル`、続いてヘッダ、空行、（あれば）ボディ、という構造です。

In [ ]:
import socket

s = socket.create_connection(("127.0.0.1", PORT))
# HTTPリクエストを自分で文字列として組み立てて送る
request_text = "GET /hello HTTP/1.1\r\nHost: localhost\r\nConnection: close\r\n\r\n"
s.sendall(request_text.encode())

# 生のレスポンスを受け取る
raw = b""
while True:
    chunk = s.recv(4096)
    if not chunk:
        break
    raw += chunk
s.close()

print(raw.decode())   # ステータス行・ヘッダ・空行・ボディ が全部テキストで見える

## 練習問題3: 生socketでステータス行を読む

`socket` で教材サーバに生のHTTP `GET /hello` を送り、レスポンスの **1行目（ステータス行）** だけを表示してください。

**ヒント**: リクエスト文字列は `"GET /hello HTTP/1.1\r\nHost: localhost\r\nConnection: close\r\n\r\n"`。受け取った全体を `\r\n` で分割した先頭がステータス行です。

In [ ]:
import socket

s = socket.create_connection(("127.0.0.1", PORT))
request_text = ____   # GET /hello の生HTTPリクエスト文字列
s.sendall(request_text.encode())

raw = b""
while True:
    chunk = s.recv(4096)
    if not chunk:
        break
    raw += chunk
s.close()

status_line = raw.decode().split("\r\n")[0]
print(status_line)   # HTTP/1.0 200 OK

# リアルタイム通信

## HTTP の限界

HTTPは **「クライアントが聞いて、サーバが答える」一問一答** です。サーバ側から自発的に「今こうなったよ」と送りつけることはできません。

チャットや対戦ゲームのように **サーバの状態が刻々と変わる** 場合、HTTPだと「変化ありましたか？」を何度も聞き続ける（**ポーリング**）しかなく、無駄が多く遅れます。

そこで登場するのが **WebSocket** です。

## WebSocket — 双方向・つなぎっぱなし

**WebSocket** は、一度つなぐと **接続を保ったまま、両方向にいつでもデータを送り合える** 通信です。

面白いのは **始まり方** です。WebSocketは最初、ふつうの **HTTPで「これからWebSocketに切り替えたい」とお願い**（`Upgrade`）します。このとき、クライアントが送った `Sec-WebSocket-Key` から、サーバは決まった計算で `Sec-WebSocket-Accept` を作って返します。これが一致して初めて接続が成立します。

計算は「鍵 + 決まった呪文(GUID)」を **SHA-1** して **base64** するだけ。前回のハッシュ・base64がそのまま出てきます。

In [ ]:
import hashlib, base64

GUID = "258EAFA5-E914-47DA-95CA-C5AB0DC85B11"   # WebSocketの決まった呪文

def websocket_accept(client_key):
    return base64.b64encode(hashlib.sha1((client_key + GUID).encode()).digest()).decode()

# 仕様書の例で確かめる
print(websocket_accept("dGhlIHNhbXBsZSBub25jZQ=="))   # s3pPLMBiTxaQ9kYGzzhZRbK+xOo=

## 練習問題4: WebSocketの `Sec-WebSocket-Accept` を計算する

クライアントから届いた `client_key` に対して、サーバが返すべき `Sec-WebSocket-Accept` を計算する関数を完成させてください。

**ヒント**: `client_key + GUID` を **SHA-1** して、その **バイト列** を base64 エンコードします。`hashlib.sha1(...).digest()` と `base64.b64encode(...)` を使います。

**期待される出力**: `s3pPLMBiTxaQ9kYGzzhZRbK+xOo=`

In [ ]:
import hashlib, base64
GUID = "258EAFA5-E914-47DA-95CA-C5AB0DC85B11"

def websocket_accept(client_key):
    return ____   # (client_key + GUID) を SHA-1 → digest → base64 → decode

print(websocket_accept("dGhlIHNhbXBsZSBub25jZQ=="))   # s3pPLMBiTxaQ9kYGzzhZRbK+xOo=

## WebSocketのフレームと「マスク」＝ XOR

WebSocketでは、データを **フレーム** という単位で送ります。そして **クライアント→サーバのデータは必ず XOR でマスク**（かく乱）される決まりです。

ここで前回の **XOR** が本当に出てきます。4バイトのマスク鍵で各バイトをXORしてあるので、**同じ鍵でもう一度XORすれば元に戻ります**（`a ^ b ^ b == a`）。

下は、マスクされたペイロードを解除する例です。

In [ ]:
def unmask(payload, mask):
    # payload の各バイトを、4バイトの mask で順番にXORする
    return bytes(payload[i] ^ mask[i % 4] for i in range(len(payload)))

mask = bytes([0x37, 0xfa, 0x21, 0x3d])
masked = bytes.fromhex("7f93")      # マスク済みのデータ（2バイト）
print("解除結果:", unmask(masked, mask).decode())   # Hi

## 練習問題5: WebSocketフレームのマスクを解く

サーバが受け取った、マスク済みのペイロード `masked` と 4バイトの `mask` があります。前回のXORを使って元のテキストに戻してください。

**ヒント**: `payload[i] ^ mask[i % 4]` を全バイtrueに対して行います。

**期待される出力**: `HELLO`

In [ ]:
masked = bytes.fromhex("cab68f76cd")
mask = bytes([0x82, 0xf3, 0xc3, 0x3a])

def unmask(payload, mask):
    return ____   # payload の各バイトを mask とXORして bytes にまとめる

print(unmask(masked, mask).decode())   # HELLO

## WebRTC — ブラウザ同士の直接通信（P2P）

HTTPやWebSocketは、必ず **サーバを経由** します。これに対し **WebRTC** は、**ブラウザ同士が直接**（Peer to Peer, P2P）データや音声・映像をやり取りする技術です。ビデオ通話などで使われています。

ただし、いきなり相手に直接つなぐことはできません。次の段取りが必要です。

1. **シグナリング**: 「私はここにいます」という接続情報を、**最初だけ何らかの方法で相手に渡す**（この受け渡し役はサーバが担う）
2. **ICE / STUN / TURN**: おたがいの「住所の候補（ICE候補）」を集めて、つながる経路を探す。家庭やオフィスは **NAT** の内側にいて直接見えないことが多く、**STUN** サーバに「私の外から見た住所は？」と聞いたり、どうしてもダメなら **TURN** サーバに中継してもらったりします
3. つながったら、あとは **直接** データが流れます

> **セキュリティの視点**: P2Pは「経路探索（ICE候補）」で自分のIPアドレスが相手に見えます。また学校や社内のネットワークは、端末どうしの直接通信を **わざと遮断**（AP分離など）していることが多く、WebRTCが繋がらないことがあります。「なぜ繋がる／繋がらないのか」を観察すること自体が、ネットワークの良い学びになります。

下のセルは、**1つのページの中で2つのブラウザを用意して互いに接続する** デモです。集まった **ICE候補** と、P2Pの通り道（DataChannel）で送り合う ping / pong が表示されます。

In [ ]:
from IPython.display import HTML, display

demo = """
<div id="log" style="font-family:monospace; white-space:pre-wrap; font-size:13px;
     background:#111; color:#0f0; padding:10px; border-radius:6px;"></div>
<script>
const log = (m) => { document.getElementById('log').textContent += m + "\n"; };
async function run() {
  const pc1 = new RTCPeerConnection();   // プレイヤーA
  const pc2 = new RTCPeerConnection();   // プレイヤーB
  // ICE候補（住所の候補）をお互いに渡す。ここでは同じページ内なので直接渡せる
  pc1.onicecandidate = e => { if (e.candidate) { log("A の ICE候補: " + e.candidate.candidate); pc2.addIceCandidate(e.candidate); } };
  pc2.onicecandidate = e => { if (e.candidate) { pc1.addIceCandidate(e.candidate); } };
  // A が直接通信路(DataChannel)を作る
  const dc = pc1.createDataChannel("chat");
  dc.onopen = () => { log("P2P接続 成立 → A が送信: ping"); dc.send("ping"); };
  dc.onmessage = e => log("A が受信: " + e.data);
  pc2.ondatachannel = e => {
    const ch = e.channel;
    ch.onmessage = m => { log("B が受信: " + m.data + " → B が pong を返信"); ch.send("pong"); };
  };
  // シグナリング（offer / answer の交換）
  const offer = await pc1.createOffer();  await pc1.setLocalDescription(offer);  await pc2.setRemoteDescription(offer);
  const answer = await pc2.createAnswer(); await pc2.setLocalDescription(answer); await pc1.setRemoteDescription(answer);
  log("シグナリング完了。ICE候補を集めています...");
}
run().catch(e => log("エラー: " + e));
</script>
"""
display(HTML(demo))

実行すると、`ICE候補: candidate:... typ host ...` のような行が並び、最後に `ping` → `pong` が流れるはずです。

- `typ host` … 自分のLAN内の住所。同じ機械・同じLANならこれで繋がります
- `typ srflx` … STUN経由で分かった「外から見た住所」（NATの外側）

**教室で試すなら**: 2人が別々の端末でブラウザ同士をP2P接続しようとすると、学校ネットワークの設定次第で **繋がらないこともあります**。そのとき ICE候補に何が出ているかを見ると、原因（NAT・AP分離など）を推測できます。この「実際に試して観察する」ことを、次の **対戦ゲーム回** につなげます。

## 練習問題6（総仕上げ）: サーバと対戦する数当てbot

サーバは `1`〜`100` の秘密の数を持っています。クライアントから `POST /guess` に `{"number": 50}` のように送ると、`{"result": "correct" / "low" / "high"}` を返します（`low` = 秘密の数はもっと大きい）。

このサーバに対して、**自動で数を当てるbot** を書いてください。当てるまでのやり取りを繰り返します。

**ヒント**: 範囲の真ん中を送り、`low` なら下限を上げ、`high` なら上限を下げる（**二分探索**）と、7回以内で必ず当たります。これは次の対戦回で作る「サーバと通信するbot」の原型です。

**期待される動作**: `N回で当たりました！ 答えは XX`（7回以内）

In [ ]:
import json, threading, time, random, urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

# --- 対戦相手のサーバ（中身は見ないでbotで当ててみよう）---
class GuessServer(BaseHTTPRequestHandler):
    secret = random.randint(1, 100)
    def do_POST(self):
        n = int(self.headers.get("Content-Length", 0))
        g = json.loads(self.rfile.read(n))["number"]
        r = "correct" if g == GuessServer.secret else ("low" if g < GuessServer.secret else "high")
        body = json.dumps({"result": r}).encode()
        self.send_response(200); self.send_header("Content-Type","application/json")
        self.end_headers(); self.wfile.write(body)
    def log_message(self, *a): pass

gs = ThreadingHTTPServer(("127.0.0.1", 0), GuessServer)
gport = gs.server_address[1]
threading.Thread(target=gs.serve_forever, daemon=True).start()
time.sleep(0.3)
URL = "http://127.0.0.1:{}/guess".format(gport)

def ask(number):
    """サーバに number を送って "correct"/"low"/"high" を受け取る"""
    req = urllib.request.Request(URL, data=json.dumps({"number": number}).encode(),
                                headers={"Content-Type": "application/json"}, method="POST")
    return json.loads(urllib.request.urlopen(req).read())["result"]

# --- ここにbotを書く（二分探索）---
low, high = 1, 100
tries = 0
while True:
    guess = ____                       # low と high の真ん中
    tries += 1
    result = ask(guess)
    if result == "correct":
        print("{}回で当たりました！ 答えは {}".format(tries, guess))
        break
    elif result == "low":              # 秘密の数はもっと大きい
        low = ____
    else:                              # もっと小さい
        high = ____